```markdown
# Agentes RAG Especializados Open Source: Guia Prático com LangChain e Qwen

Este notebook ensina como construir um assistente virtual inteligente capaz de responder a dúvidas técnicas de um produto utilizando a técnica de **RAG (Retrieval-Augmented Generation)**. Através de ferramentas open-source e modelos locais, você aprenderá a carregar um manual em PDF, processar e indexar as informações em um banco de dados vetorial e criar um agente conversacional com memória de sessão capaz de interagir em linguagem natural.

## 🎯 Objetivos de Aprendizagem
Ao final deste guia, você será capaz de:
* Compreender o conceito e a arquitetura de um sistema RAG (Geração Aumentada de Recuperação).
* Carregar e extrair dados de documentos complexos (como manuais em PDF).
* Criar representações vetoriais (*embeddings*) de textos e armazená-los em um banco vetorial local (FAISS).
* Configurar e executar um Modelo de Linguagem Grande (LLM) local integrado ao pipeline do LangChain.
* Implementar controle de histórico e memória para manter conversas contínuas por sessão.

---

## 📚 Bibliotecas Utilizadas e Pré-requisitos

Para que o nosso sistema funcione, precisamos instalar as seguintes ferramentas:
* **langchain** (e suas extensões `community`, `core` e `classic`): O framework principal que conecta nosso modelo de linguagem aos dados externos e gerencia o fluxo da conversa.
* **transformers** e **accelerate**: Bibliotecas da Hugging Face essenciais para carregar, otimizar e rodar modelos de deep learning de forma rápida e eficiente na GPU.
* **sentence-transformers**: Usada para gerar os *embeddings*, transformando palavras e frases em vetores numéricos de significado.
* **faiss-cpu**: Uma biblioteca extremamente rápida para busca de similaridade e armazenamento de vetores estruturados.
* **pypdf**: Necessária para extrair textos brutos de arquivos no formato PDF.
```

In [1]:
!pip install -U langchain langchain-classic langchain-community langchain-core langchain-huggingface transformers accelerate sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.1/591.1 kB 5.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 7.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 14.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 13.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 23.5 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.20
    Uninstalling langchain-protocol-0.0.20:
      Successfully uninstalled langchain-protocol-0.0.20
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7

```markdown
## 🛠️ Passo 1: Importação dos Módulos Necessários

### O que este passo faz:
Importamos as ferramentas necessárias para construir a arquitetura RAG. Isso inclui o suporte a tensores com PyTorch, conectores para o modelo Hugging Face, o banco de dados vetorial FAISS, o leitor de PDF e os componentes de histórico e estrutura de prompts do LangChain.

### Por que ele é necessário:
Como o Python por si só não possui essas funcionalidades nativas, precisamos trazer esses pacotes para o ambiente a fim de montar nossa cadeia de execução.

### Como interpretar o resultado/saída:
Se o código for executado sem erros, o ambiente está pronto. Note que pode haver um aviso (*DeprecationWarning*) sobre a biblioteca `langchain-community` estar passando por reestruturações. Isso é normal no ecossistema ativo de IA e não impede o funcionamento do código.
```

In [2]:
import torch
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline, HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import ConversationalRetrievalChain

/tmp/ipykernel_550/4221278709.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


```markdown
## 📄 Passo 2: Carregamento do Documento (Manual PDF)

### O que este passo faz:
Utiliza o `PyPDFLoader` para ler o arquivo `manual_avancado_nobreak_fxp2000.pdf` e extrair todo o seu texto de forma estruturada, dividindo-o por páginas.

### Por que ele é necessário:
O modelo de linguagem (LLM) não sabe o que está escrito no manual do produto por padrão. Para que ele responda perguntas com precisão, precisamos primeiro ler esse arquivo externo e convertê-lo em um formato que o código consiga manipular.

### Como interpretar o resultado/saída:
A execução bem-sucedida cria uma lista de documentos (`documents`) onde cada elemento contém o conteúdo textual de uma página e seus metadados (como o número da página).
```

In [3]:
loader = PyPDFLoader("manual_avancado_nobreak_fxp2000.pdf")
documents = loader.load()

```markdown
## 🧠 Passo 3: Criação de Embeddings e Banco de Vetores (FAISS)

### Termo Técnico Importante:
* **Embeddings**: São representações numéricas (vetores) que capturam o significado semântico de um texto. Frases com significados parecidos geram vetores que ficam próximos no espaço geométrico.
* **Vector Store (Banco Vetorial)**: Banco de dados especializado em armazenar e buscar esses vetores de forma extremamente rápida.

### O que este passo faz:
Instancia o modelo `BAAI/bge-m3` para gerar os embeddings do manual extraído e os armazena no FAISS. Em seguida, define uma ferramenta de busca (`retriever`) que nos ajudará a resgatar os trechos do manual mais relevantes para qualquer pergunta feita.

### Por que ele é necessário:
Em vez de enviar todo o manual para a IA (o que seria lento e caro), transformamos o manual em vetores. Quando o usuário faz uma pergunta, o sistema calcula o vetor da pergunta e busca apenas as partes do manual que são semanticamente parecidas com ela.

### Como interpretar o resultado/saída:
Aparecerá um aviso da Hugging Face sobre requisições não autenticadas (caso você não use um token). Isso é apenas informativo. O banco de dados vetorial foi criado com sucesso na memória do Colab.
```

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (7 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

```markdown
## 🤖 Passo 4: Inicialização do Modelo de Linguagem (LLM)

### O que este passo faz:
Carrega o modelo de linguagem open-source `Qwen/Qwen3-4B-Instruct-2507` diretamente na memória da máquina (usando GPU via `device_map="auto"` e precisão de float16 para economizar memória), encapsulando-o no ecossistema do LangChain por meio do `ChatHuggingFace`.

### Por que ele é necessário:
Precisamos de um "cérebro" de inteligência artificial para ler as perguntas do usuário, ler o contexto recuperado do manual e formular uma resposta fluida e coerente em português.

### Como interpretar o resultado/saída:
O download do modelo e dos pesos será exibido em barras de progresso. Após a conclusão, a IA estará carregada na memória de vídeo (VRAM) da sua GPU do Colab.
```

In [5]:
llm = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen3-4B-Instruct-2507",
    task="text-generation",
    device_map="auto",
    model_kwargs={"dtype": torch.float16},
    pipeline_kwargs={"max_new_tokens": 512, "return_full_text": False},
)
chat = ChatHuggingFace(llm=llm)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

```markdown
## 📝 Passo 5: Definição do Prompt de Sistema

### O que este passo faz:
Cria um modelo de prompt instruindo a IA sobre qual papel ela deve desempenhar (um assistente de atendimento especialista no nobreak FCP-2000 [VERIFICAR]) e como ela deve se comportar utilizando o contexto recuperado.

### Por que ele é necessário:
Sem essa diretriz, o modelo poderia inventar respostas baseadas em informações gerais da internet ou responder a assuntos não relacionados ao produto. O prompt limita o escopo de atuação e garante respostas profissionais.

💡 **Para pensar**:
O manual carregado e o nome do arquivo se referem ao modelo **FXP-2000**. No entanto, a instrução do sistema no prompt está configurada com a sigla **FCP-2000**. Isso pode gerar respostas inconsistentes se o usuário perguntar pela sigla exata do produto. Seria ideal alinhar essa nomenclatura!
```

In [6]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
Você é um assistente virtual especializado no atendimento ao cliente no nobreak FCP-2000.
Responda claramente perguntas técnicas, funcionalidades, garantia, manutenção, atualização e suporte técnico.
Se a pergunta for irrelevante, responda educadamente recusando a pergunta.

Use o contexto abaixo para responder:
{context}
"""),
    ("human", "{question}"),
])

```markdown
## 🔗 Passo 6: Criação da Cadeia de Conversação com Histórico

### O que este passo faz:
Instancia uma cadeia do tipo `ConversationalRetrievalChain` unindo a LLM, o recuperador de documentos e o prompt. Para dar capacidade de conversa contínua, envolvemos essa cadeia no `RunnableWithMessageHistory`, usando um dicionário simples (`store`) para salvar o histórico de mensagens de cada sessão.

### Por que ele é necessário:
Uma IA por si só não lembra do que foi dito na mensagem anterior. Para termos um chat real (onde você diz "O que significa esse modelo?" após ter falado dele antes), precisamos de um componente que anexe as perguntas e respostas passadas ao contexto da nova interação.

### Como interpretar o resultado/saída:
Aparecerão avisos de depreciação sugerindo migração para o LangGraph no futuro. O código ainda funciona perfeitamente para este exemplo educacional.
```

In [7]:
# memória por sessão "chat_id"
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=chat,
    retriever=retriever,
    return_source_documents=False,
    combine_docs_chain_kwargs={"prompt": prompt},
)

chain_with_history = RunnableWithMessageHistory(
    qa_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


```markdown
## 💬 Passo 7: Interação por Chat de Testes

### O que este passo faz:
Inicia um loop infinito (`while True`) que pede entradas do usuário no terminal do notebook, envia as mensagens para a cadeia com a identificação de sessão `"sessao1"`, e exibe a resposta gerada. O loop é encerrado se o usuário digitar "sair", "quit" ou "exit".

### Por que ele é necessário:
É a interface de testes prática para interagir diretamente com o assistente criado e validar se ele está recuperando as informações do PDF corretamente e mantendo o contexto.

### Como interpretar o resultado/saída:
Ao executar a célula, um campo de texto surgirá. Você poderá fazer perguntas como "me fale um pouco sobre o nobreak" e o assistente responderá utilizando dados do PDF.

💡 **Para pensar**:
O loop utiliza o método `input()` interativo do Python. Em produção ou ao criar uma interface web (ex. usando Streamlit), a captura de dados e o gerenciamento de sessões seriam feitos por meio de APIs ou WebSockets, mantendo o histórico em bancos de dados persistentes como Redis, e não na memória volátil (`store`).
```

In [8]:
# Testes
session_id = "sessao1"
while True:
    pergunta = input("Você: ")
    if pergunta.lower() in ["sair", "quit", "exit"]:
        print("vlw, Até logo")
        break
    resposta = chain_with_history.invoke(
        {"question": pergunta},
        config={"configurable": {"session_id": session_id}},
    )
    print("Assistente:", resposta["answer"])

Você: Você: me fale um pouco sobre o nobreak, eu consigo usar ele aonde?


/tmp/ipykernel_550/1605276477.py:6: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id] = InMemoryChatMessageHistory()


Assistente: Claro! Aqui está um resumo sobre o **Nobreak FXP-2000** e onde você pode usá-lo:

---

### 🔌 O que é o Nobreak FXP-2000?
O Nobreak FXP-2000 é um equipamento de proteção contra interrupções na energia elétrica, projetado para garantir que seus dispositivos continuem funcionando mesmo quando a rede elétrica falhar ou apresentar instabilidades (como sobretensão, subtensão ou falta de energia).

Ele oferece **autonomia de até 25 minutos a 50% da carga**, com potência nominal de **2000VA** (equivalente a cerca de 1500W de potência ativa), e é ideal para ambientes onde a estabilidade da energia é essencial.

---

### 🏠 Onde você pode usar o Nobreak FXP-2000?

✅ **Ambientes domésticos**  
- Computadores de alto desempenho  
- TVs e consoles de videogame  
- Roteadores, impressoras e dispositivos de rede  
- Sistemas de automação residencial (como luzes, câmeras, controles)  
- Dispositivos de armazenamento (como SSDs ou servidores domésticos)

✅ **Ambientes corporativos pequenos**

```markdown
## ⚡ Passo 8: Verificação de Hardware (GPU)

### O que este passo faz:
Verifica se a aceleração por hardware por meio da GPU (CUDA) está ativada e confirma em qual dispositivo o pipeline do modelo está rodando.

### Por que ele é necessário:
Modelos de Linguagem Grandes (LLMs) são pesados e lentos para rodar em CPU comum. Garantir que o modelo está rodando no dispositivo `cuda:0` (GPU) garante respostas em segundos em vez de minutos.

### Como interpretar o resultado/saída:
Se retornar `True` e `cuda:0`, seu modelo está acelerado de forma otimizada pela GPU fornecida pelo Google Colab.
```

```markdown
---

## ⚠️ Erros Comuns e Dicas
1. **Erro de Memória Estourada (Out Of Memory - OOM)**: Se você tentar rodar o notebook sem selecionar um ambiente de execução com GPU (T4 ou superior no Colab), o carregamento do modelo Qwen falhará ou ficará extremamente lento. Vá em *Ambiente de Execução > Alterar tipo de ambiente de execução* e escolha **GPU T4**.
2. **Arquivo Não Encontrado**: Se o arquivo `manual_avancado_nobreak_fxp2000.pdf` não for carregado no diretório atual do Colab antes da execução do passo 2, o código resultará em erro. Certifique-se de realizar o upload do arquivo usando o painel lateral de arquivos do Colab.
3. **Uso de Versões Antigas**: O LangChain é uma biblioteca que se atualiza muito rapidamente. Usar importações antigas pode gerar diversos *DeprecationWarnings*. Fique atento à documentação oficial para atualizar a sintaxe quando necessário.

---

## 🎓 Resumo Final
Neste notebook você aprendeu:
* Como estruturar uma aplicação RAG local usando componentes do ecossistema Hugging Face e LangChain.
* O papel essencial dos **Embeddings** e **Bancos Vetoriais** para fornecer conhecimentos externos sem precisar treinar novamente o modelo.
* Como o gerenciamento de sessão de chat permite criar experiências conversacionais contínuas que lembram o contexto passado.

---

## 📝 Exercícios Propostos
1. **[Fácil]** Altere o prompt do sistema para que o assistente sempre finalize as respostas perguntando educadamente se o usuário precisa de mais alguma informação sobre o produto.
2. **[Médio]** Altere o código para que ele leia um arquivo de texto simples (`.txt`) ou uma página da web em vez do manual em PDF, adaptando o componente de carregamento do documento (`Loader`).
3. **[Desafiador]** Implemente uma segunda sessão de testes (`sessao2`) rodando em paralelo no loop para verificar se o histórico de uma conversa não vaza ou interfere nas respostas da outra sessão.

---

## 🚀 Para Ir Além
* Pesquise sobre **Text Splitters (Chunking)**: Como dividir documentos gigantescos em pequenos pedaços estruturados de forma inteligente para melhorar a busca de similaridade.
* Estude sobre **Quantização**: Técnicas de compressão de modelos (ex. carregar modelos em 4-bits ou 8-bits) que permitem rodar LLMs ainda maiores em hardwares limitados.
* Explore o **LangGraph**: O framework de nova geração para criação de agentes inteligentes com fluxos cíclicos complexos e controle fino de memória persistente.
```